# Before your start:
- Read the README.md file
- Comment as much as you can and use the resources (README.md file)
- Happy learning!

In [1]:
# import numpy and pandas
import pandas as pd
import numpy as np
from scipy.stats import trim_mean, mode, skew, gaussian_kde, pearsonr, spearmanr, beta
from statsmodels.stats.weightstats import ztest as ztest

from scipy.stats import ttest_ind, norm, t
from scipy.stats import f_oneway
from scipy.stats import sem

# Challenge 1 - Exploring the Data

In this challenge, we will examine all salaries of employees of the City of Chicago. We will start by loading the dataset and examining its contents

In [2]:
# Run this code:
salaries = pd.read_csv('../data/Current_Employee_Names__Salaries__and_Position_Titles.csv')

Examine the `salaries` dataset using the `head` function below.

In [3]:
# Preview the first five rows of the dataset.
salaries.head()


,Name,Job Titles,Department,Full or Part-Time,Salary or Hourly,Typical Hours,Annual Salary,Hourly Rate
0,"AARON, JEFFERY M",SERGEANT,POLICE,F,Salary,NaN,101442.0,NaN
1,"AARON, KARINA",POLICE OFFICER (ASSIGNED AS DETECTIVE),POLICE,F,Salary,NaN,94122.0,NaN
2,"AARON, KIMBERLEI R",CHIEF CONTRACT EXPEDITER,GENERAL SERVICES,F,Salary,NaN,101592.0,NaN
3,"ABAD JR, VICENTE M",CIVIL ENGINEER IV,WATER MGMNT,F,Salary,NaN,110064.0,NaN
4,"ABASCAL, REECE E",TRAFFIC CONTROL AIDE-HOURLY,OEMC,P,Hourly,20.0,NaN,19.86


# Challenge 2 - Hypothesis Tests

In this section of the lab, we will test whether the hourly wage of all hourly workers is significantly different from $30/hr. Use only the hourly workers (`Salary or Hourly` is `Hourly`) and their `Hourly Rate`. Import the correct one sample test function from scipy and perform a two-sided test at the 5% significance level. Write down H0 and H1 first, then state your conclusion.

In [4]:
from scipy.stats import ttest_1samp

# H0: The population mean hourly wage is $30/hr (mu = 30).
# H1: The population mean hourly wage differs from $30/hr (mu != 30).
alpha = 0.05
hourly_wages = salaries.loc[
    salaries["Salary or Hourly"].eq("Hourly"), "Hourly Rate"
].dropna()

test_result = ttest_1samp(hourly_wages, popmean=30, alternative="two-sided")
print(f"Hourly workers: {len(hourly_wages):,}")
print(f"Sample mean: ${hourly_wages.mean():.2f}/hr")
print(f"t-statistic: {test_result.statistic:.3f}")
print(f"p-value: {test_result.pvalue:.4g}")

if test_result.pvalue < alpha:
    print("Reject H0: the mean hourly wage is significantly different from $30/hr.")
else:
    print("Fail to reject H0: there is not enough evidence that the mean differs from $30/hr.")


Hourly workers: 8,022
Sample mean: $32.79/hr
t-statistic: 20.620
p-value: 4.323e-92
Reject H0: the mean hourly wage is significantly different from $30/hr.


# Challenge 3 - Constructing Confidence Intervals

While testing our hypothesis is a great way to gather empirical evidence for accepting or rejecting the hypothesis, another way to gather evidence is by creating a confidence interval. A confidence interval gives a range of plausible values for the true mean of the population. A 95% confidence interval comes from a method that, over many repeated samples, captures the true mean 95% of the time.

To read more about confidence intervals, click [here](https://en.wikipedia.org/wiki/Confidence_interval).


In the cell below, we will construct a 95% confidence interval for the mean hourly wage of all hourly workers. 

The confidence interval is computed in SciPy using the `t.interval` function. You can read more about this function [here](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.t.html).

To compute the confidence interval of the hourly wage, use the 0.95 for the confidence level, number of rows - 1 for degrees of freedom, the mean of the sample for the location parameter (`loc`) and the standard error for the scale (`scale`). The standard error can be computed using [this](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.sem.html) function in SciPy.

In [5]:
# A 95% t confidence interval for the population mean hourly wage.
confidence_interval = t.interval(
    confidence=0.95,
    df=len(hourly_wages) - 1,
    loc=hourly_wages.mean(),
    scale=sem(hourly_wages),
)
print(
    "95% confidence interval for the mean hourly wage: "
    f"(${confidence_interval[0]:.2f}, ${confidence_interval[1]:.2f})"
)


95% confidence interval for the mean hourly wage: ($32.52, $33.05)


# Challenge 4 - Hypothesis Tests of Proportions

Another type of one sample test is a hypothesis test of proportions. In this test, we examine whether the proportion of a group in our sample is significantly different than a fraction. 

You can read more about one sample proportion tests [here](http://sphweb.bumc.bu.edu/otlt/MPH-Modules/BS/SAS/SAS6-CategoricalData/SAS6-CategoricalData2.html).

In the cell below, use the `proportions_ztest` function from `statsmodels` to perform a hypothesis test that will determine whether the proportion of hourly workers in the City of Chicago is significantly different from 25%, at the 5% significance level.

In [6]:
from statsmodels.stats.proportion import proportions_ztest

# H0: 25% of employees are hourly workers (p = 0.25).
# H1: The proportion of hourly workers differs from 25% (p != 0.25).
alpha = 0.05
pay_type = salaries["Salary or Hourly"]
count_hourly = pay_type.eq("Hourly").sum()
number_of_employees = pay_type.notna().sum()
observed_proportion = count_hourly / number_of_employees

z_statistic, p_value = proportions_ztest(
    count=count_hourly,
    nobs=number_of_employees,
    value=0.25,
    alternative="two-sided",
)
print(f"Hourly employees: {count_hourly:,} of {number_of_employees:,} ({observed_proportion:.2%})")
print(f"z-statistic: {z_statistic:.3f}")
print(f"p-value: {p_value:.4g}")

if p_value < alpha:
    print("Reject H0: the proportion of hourly workers is significantly different from 25%.")
else:
    print("Fail to reject H0: there is not enough evidence that the proportion differs from 25%.")


Hourly employees: 8,022 of 33,183 (24.18%)
z-statistic: -3.510
p-value: 0.0004481
Reject H0: the proportion of hourly workers is significantly different from 25%.
